# Code mẫu — Regression và ranking metrics

“Đoán sai bao nhiêu phút” và “bài phù hợp được xếp ở hạng mấy” là hai câu hỏi khác nhau, nên cần hai loại metric khác nhau. Các ví dụ dưới đây tính công thức bằng tay rồi đối chiếu với thư viện, sau đó tự cài một scorer cho ranking với quy ước rõ ràng, và kiểm nó trên những danh sách nhỏ.

## Ví dụ 1: Một lỗi lớn làm hai metric chọn hai model khác nhau

Ba bài luyện thực tế mất 5, 10 và 15 phút, còn dự đoán là 6, 9 và 18 phút. Ta tính mọi metric từ cùng một vector phần dư. Sau đó so phương án A, chỉ sai một lần nhưng sai 6 phút, với phương án B, lần nào cũng sai đều 3 phút.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

y = np.array([5.0, 10.0, 15.0])
prediction = np.array([6.0, 9.0, 18.0])
residual = prediction - y
manual_mae = np.abs(residual).mean()
manual_mse = (residual ** 2).mean()
assert np.isclose(manual_mae, mean_absolute_error(y, prediction))
assert np.isclose(manual_mse, mean_squared_error(y, prediction))
print("Phần dư, phút:", residual.tolist())
print(f"MAE={manual_mae:.3f} phút, MSE={manual_mse:.3f} phút²")
print(f"RMSE={root_mean_squared_error(y, prediction):.3f} phút, R2={r2_score(y, prediction):.3f}")
for name, pred in [("A", y + [0, 0, 6]), ("B", y + [3, 3, 3])]:
    print(f"{name}: MAE={mean_absolute_error(y, pred):.3f}, RMSE={root_mean_squared_error(y, pred):.3f}")

Phần dư, phút: [1.0, -1.0, 3.0]
MAE=1.667 phút, MSE=3.667 phút²
RMSE=1.915 phút, R2=0.780
A: MAE=2.000, RMSE=3.464
B: MAE=3.000, RMSE=3.000


Phần dư `1, −1, 3` cho MAE bằng `5/3` và MSE bằng `11/3`. RMSE lấy căn **sau** khi đã lấy trung bình bình phương, nên nó có lại đơn vị phút. A có MAE thấp hơn B, nhưng RMSE lại cao hơn, vì một lỗi 6 phút góp tới 36 vào tổng bình phương. Nói cách khác, chọn metric là chọn loại lỗi mình sợ hơn: MAE coi mọi phút sai như nhau, còn RMSE phạt nặng những lần sai lớn. R² bằng 0.78 cũng không có nghĩa là đoán đúng 78% số bài.

## Ví dụ 2: Đổi loss là đổi luôn baseline hằng số tốt nhất

Thời gian làm bốn bài trong train là `2, 2, 2, 10` phút. Giá trị 10 kéo mean lên, nhưng không làm median thay đổi. Ta dùng DummyRegressor để xem hai baseline trên chính train, nhằm hiểu mỗi loss chọn hằng số nào. Đây chưa phải phép đánh giá trên dữ liệu mới.

In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = np.zeros((4, 1))
y_train = np.array([2.0, 2.0, 2.0, 10.0])
for strategy in ["mean", "median"]:
    model = DummyRegressor(strategy=strategy).fit(X_train, y_train)
    prediction = model.predict(X_train)
    print(f"{strategy}: hằng số={prediction[0]:.1f}, MAE={mean_absolute_error(y_train, prediction):.1f}, MSE={mean_squared_error(y_train, prediction):.1f}")

mean: hằng số=4.0, MAE=3.0, MSE=12.0
median: hằng số=2.0, MAE=2.0, MSE=16.0


Đoán hằng số 4, tức mean, cho MSE 12, thấp hơn MSE 16 của median. Nhưng theo MAE thì median lại thắng, 2 so với 3. Đây đúng là điều bài lý thuyết đã nói: MSE nhỏ nhất ở mean, còn MAE nhỏ nhất ở median. Khi làm bài thật, hằng số vẫn phải học từ train, và việc chọn giữa các phương án phải dựa trên metric của đề, chấm trên validation.

## Ví dụ 3: Cùng sai một phút, nhưng tỉ lệ sai khác hẳn nhau

Hai bài mất 1 phút và 100 phút, và đều bị dự đoán dư một phút. MAPE lấy chính target làm mẫu số, nên cùng một phút sai lại có trọng lượng rất khác nhau. Ta thử thêm một target bằng 0 để thấy vì sao phải kiểm miền dữ liệu trước khi dùng metric này.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

y = np.array([1.0, 100.0])
prediction = np.array([2.0, 101.0])
relative_errors = np.abs(prediction - y) / np.abs(y)
print("Lỗi tương đối từng bài:", relative_errors.tolist())
print(f"MAE={mean_absolute_error(y, prediction):.2f} phút, MAPE={100 * mean_absolute_percentage_error(y, prediction):.2f}%")
zero_y = np.array([0.0, 10.0])
zero_prediction = np.array([1.0, 10.0])
print("Có target 0:", bool(np.any(zero_y == 0)))
print(f"MAPE thư viện với target 0={mean_absolute_percentage_error(zero_y, zero_prediction):.3e}")

Lỗi tương đối từng bài: [1.0, 0.01]
MAE=1.00 phút, MAPE=50.50%
Có target 0: True
MAPE thư viện với target 0=2.252e+15


Lỗi tương đối của hai bài là 100% và 1%, nên MAPE ra 50.5%, dù MAE chỉ là một phút. Bài ngắn chiếm gần hết điểm phạt. Khi có target bằng 0, thư viện dùng một mẫu số cực nhỏ thay cho 0, nên trả về một con số khổng lồ, và con số ấy không còn ý nghĩa gì. Nếu đề có scorer chính thức, hãy làm đúng theo quy ước của nó, đừng tự cộng thêm 1 vào target cho đỡ lỗi.

## Ví dụ 4: Từ score tạo ranking, rồi tự tính AP@3

Có hai yêu cầu gợi ý là `Q1` và `Q2`, mỗi yêu cầu có tập bài phù hợp riêng. Score chỉ dùng để sắp thứ tự, nếu bằng nhau thì xếp theo mã bài. Quy ước AP@k ở đây là chia cho `min(số bài phù hợp, k)`, yêu cầu không có bài phù hợp nào thì được 0, và danh sách có bài trùng trong top k thì bị từ chối.

In [ ]:
import numpy as np

def ap_at_k(ranking, relevant, k):
    if k < 1:
        raise ValueError("k phải dương")
    top = list(ranking[:k])
    if len(top) != len(set(top)):
        raise ValueError("Duplicate trong top k")
    relevant = set(relevant)
    if not relevant:
        return 0.0
    hits = 0
    contribution = 0.0
    for position, item in enumerate(top, start=1):
        if item in relevant:
            hits += 1
            contribution += hits / position  # Chỉ cộng precision tại những vị trí trúng bài phù hợp.
    return contribution / min(len(relevant), k)

scores = {
    "Q1": {"P1": 0.9, "P2": 0.8, "P3": 0.7, "P4": 0.2},
    "Q2": {"P1": 0.9, "P2": 0.8, "P3": 0.2},
}
relevant = {"Q1": {"P1", "P3"}, "Q2": {"P2"}}
ap_scores = []
for query, item_scores in scores.items():
    ranking = sorted(item_scores, key=lambda item: (-item_scores[item], item))[:3]
    ap = ap_at_k(ranking, relevant[query], 3)
    ap_scores.append(ap)
    print(f"{query}: top3={ranking}, AP@3={ap:.6f}")
assert np.isclose(ap_scores[0], (1 + 2/3) / 2)
assert np.isclose(ap_scores[1], 1/2)
print(f"MAP@3={np.mean(ap_scores):.6f}")

improved = ["P1", "P3", "P2"]
print(f"Q1 đưa hit lên hạng 2: AP@3={ap_at_k(improved, relevant['Q1'], 3):.6f}")
assert ap_at_k(["P1"], {"P1", "P3"}, 3) == 0.5
assert ap_at_k(["P1"], set(), 3) == 0.0
try:
    ap_at_k(["P1", "P1", "P2"], relevant["Q1"], 3)
except ValueError as error:
    print("Chặn bảng sai:", error)

Q1: top3=['P1', 'P2', 'P3'], AP@3=0.833333
Q2: top3=['P1', 'P2', 'P3'], AP@3=0.500000
MAP@3=0.666667
Q1 đưa hit lên hạng 2: AP@3=1.000000
Chặn bảng sai: Duplicate trong top k


Q1 trúng ở hạng 1 và hạng 3, nên AP là `(1 + 2/3) / 2 = 5/6`. Q2 chỉ trúng ở hạng 2, nên AP là 0.5. MAP là trung bình **theo từng yêu cầu**, bằng `2/3`. Đưa P3 lên hạng 2 làm AP của Q1 thành 1, dù số bài trúng trong top 3 vẫn là hai. Ranking thưởng cho việc đặt bài đúng lên sớm. Mấy dòng assert cuối khóa chặt những quy tắc dễ cãi nhau: danh sách ngắn hơn k, yêu cầu không có bài phù hợp, và bài bị trùng. Một scorer dùng mẫu số khác sẽ cho kết quả khác, nên khi đi thi phải đọc kỹ cách đề định nghĩa.